# SSA Analysis Workbench

This notebook is a practical entrypoint for `ssa.analysis` over logs in `logs/`.

It covers:
- run discovery + run-level index (`ssa.analysis.io`)
- agent-level summaries (`ssa.analysis.metrics`)
- trace extraction (`ssa.analysis.traces`)
- directional claim checks (`ssa.analysis.rebuttal`)

In [1]:
from pathlib import Path

import pandas as pd

from ssa.analysis import (
    batch_extract_traces,
    build_run_index,
    discover_logs,
    evaluate_rebuttal_claims,
    get_summary_df,
)
from ssa.common import ExperimentLog

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path("logs")
print(f"Log root: {ROOT.resolve()}")

Log root: /Users/chris/ssa_submission/logs


In [ ]:
paths = discover_logs(ROOT)
print(f"Found {len(paths)} log files")
paths[:10]

In [ ]:
run_df = build_run_index(root=ROOT, output_path=ROOT / "run_index.csv")
print(f"run_index rows: {len(run_df)}")
run_df.head(20)

In [ ]:
if run_df.empty:
    print("No runs to summarize yet. Generate logs first via ssa.run_experiment.")
else:
    variant_summary = (
        run_df.groupby(["study", "variant"], dropna=False)
        .agg(
            n_runs=("path", "count"),
            mean_train_rate=("train_rate", "mean"),
            mean_bid_ratio=("mean_winning_bid_ratio", "mean"),
            mean_ssa_adv=("ssa_minus_control_reward", "mean"),
            mean_tokens=("total_tokens", "mean"),
        )
        .reset_index()
        .sort_values(["study", "variant"])
    )
    display(variant_summary)

In [ ]:
if not paths:
    print("No logs available for per-agent summary.")
else:
    sample_path = paths[0]
    exp_log = ExperimentLog.load(str(sample_path))
    agent_df = get_summary_df(exp_log, fp=sample_path.as_posix())
    print(f"Loaded sample run: {sample_path}")
    display(agent_df.sort_values("rank"))

In [ ]:
if not paths:
    print("No logs available for trace analysis.")
else:
    # Load a bounded number for speed in interactive analysis.
    max_logs = 20
    selected = paths[:max_logs]
    run_logs = {p.as_posix(): ExperimentLog.load(str(p)) for p in selected}
    trace_df = batch_extract_traces(run_logs)
    print(f"Extracted {len(trace_df)} trace rows from {len(selected)} runs")
    if len(trace_df):
        display(trace_df.head(10))
        action_counts = (
            trace_df.groupby(["run", "action"], dropna=False)
            .size()
            .rename("n")
            .reset_index()
            .sort_values(["run", "n"], ascending=[True, False])
        )
        display(action_counts.head(30))

In [ ]:
claim_df = evaluate_rebuttal_claims(run_df)
claim_df

In [2]:
from ssa.analysis import get_summary_df

In [18]:
# Baseline experiments
import os
logs = []
for f in os.listdir('logs/rebuttal/ssa_scaffold_50'):
    try:
        logs.append((f,ExperimentLog.load(f"logs/rebuttal/ssa_scaffold_50/{f}")))
    except:
        print(f)
        continue
   

analysis


In [19]:
summary_dfs = [get_summary_df(l, ix) for ix, l in logs]
for df in summary_dfs:
    df['atype'] = df['agent_id'].apply(lambda x: x[:-1])

In [20]:
df = pd.concat(summary_dfs)
train_target = df.query('train_target > 0').groupby('atype').train_target.mean()
df['recovery'] = df['recovery'] * 100
df['train_p'] = df['train_p'] * 100
df = df.rename({'reward_normalized': "market_share"}, axis=1)
df['market_share'] = df['market_share'] * 100
df['winrate'] = df['winrate'] * 100
df['rep_max'] *= 5
df['rep_avg'] *= 5
df = df.groupby('atype').mean(numeric_only=True)
df['train_target'] = train_target

df = df.sort_values('rank')
df


,reward,market_share,rank,winrate,win_prio,recovery,rank_jump,top_base_price,avg_base_price,all_bids,winning_bids,train_p,train_target,skill_sum,skill_max,skill_spec,rep_avg,rep_max,rep_spec,total_tokens,completion_tokens
atype,,,,,,,,,,,,,,,,,,,,,
SSAS-,176.021940,6.696316,8.575000,38.501664,2.198201,14.362245,11.281250,6.891265,6.773098,0.733655,0.708974,9.538048,1.050847,249.98750,83.0750,0.031152,2.449076,4.035942,0.029546,280050.8625,16855.5375
SSAD-,156.836453,5.870659,8.934375,38.121830,2.225096,13.265306,11.462500,7.032800,6.763285,0.720833,0.697716,12.077796,1.602564,252.44375,82.4750,0.029877,2.477412,3.932941,0.026337,257349.1500,8907.4750
COT-,131.706580,4.949976,10.062500,29.447545,2.253196,13.801020,12.125000,7.093050,6.877857,0.743661,0.718380,10.132696,1.031250,247.11250,81.2125,0.029645,2.433089,3.910905,0.026561,248151.7500,8569.0875
POL-,28.355060,1.074927,17.829167,6.383474,2.817400,4.183673,9.995833,9.676610,6.996575,0.883333,0.880808,17.526202,1.294118,255.95000,81.3750,0.028059,2.474497,3.632199,0.018972,0.0000,0.0000


## Next Steps

- Filter by study/variant with `discover_logs(ROOT, study=..., variant=...)`.
- Export tables with `to_csv(...)` for paper figures.
- For script-first workflows, see `scripts/analysis/*.py` (same underlying APIs used here).